
# Step 5-1. Multimodal T2D Prediction — XGBoost

## 1. Overview

This notebook demonstrates participant-level type 2 diabetes (T2D) prediction using XGBoost.

The model integrates 4,100 predictors prepared in Step 4:

- ABD-US GNN activation vectors: 1,024 features
- BMD left hip DenseNet121 activation vectors: 1,024 features
- BMD right hip DenseNet121 activation vectors: 1,024 features
- BMD spine DenseNet121 activation vectors: 1,024 features
- Age, sex, and family history: 3 features
- Equal-weighted PRS-CSx score: 1 feature

The PRS-CSx score was combined using predefined 1:1:1 coefficients and scaled using the Training1 and Training2 participants, as described in Step 4.

The demonstration uses the existing dataset partitions without reallocating participants.

## 2. Dataset Partition

| Dataset | N | Role |
|---|---:|---|
| Training1 | 32 | Used previously for DenseNet training; excluded from XGBoost training |
| Training2 | 32 | XGBoost model training |
| Validation | 16 | Early stopping and decision-threshold selection |
| Testing | 20 | Final demonstration evaluation |

The Testing dataset is not used to fit the XGBoost model, select the number of boosting rounds, or determine the classification threshold.

## 3. Model Architecture and Hyperparameters

| Hyperparameter | Original implementation | GitHub demo |
|---|---|---|
| Architecture | XGBoost classifier | XGBoost classifier |
| Objective | Binary logistic | Binary logistic |
| Maximum tree depth | 6 (default) | 6 |
| Learning rate | 0.3 (default) | 0.3 |
| Maximum boosting rounds | 100 (default) | 100 |
| L1 regularization | 0 (default) | 0 |
| L2 regularization | 1 (default) | 1 |
| Evaluation metric | AUC | AUC |
| Early stopping | 30 rounds | 30 rounds |
| Model selection | Highest validation AUC | Highest validation AUC |
| Classification threshold | Validation Youden Index | Validation Youden Index |
| Training dataset | Training2 | Training2 |

The original implementation used XGBoost 1.7.6. The demonstration uses XGBoost 2.1.4 with an updated GPU interface for compatibility with the current software environment.

All parameters are defined explicitly in the configuration cell.

## 4. Evaluation

Model performance is summarized using AUC, Accuracy, Sensitivity, Specificity, Precision, NPV, F1 Score, Youden Index, Average Precision, and Brier Score.

The classification threshold is chosen on Validation by maximizing Sensitivity + Specificity (equivalent to maximizing the Youden Index).

The selected threshold is then applied unchanged to Testing.

A confusion matrix is provided separately for Validation and Testing.

Due to the small demonstration sample size, performance estimates should not be interpreted as estimates of the original study's predictive performance.

## 5. Environment Setup

This notebook uses the shared `T2D GPU` environment defined in `environments/gpu.yml`.

From the downloaded repository root, new users can run:

```bash
conda env create -f environments/gpu.yml

conda run -n t2d-gpu python -m ipykernel install \
    --user --name t2d-gpu --display-name "T2D GPU"
```

Open this notebook and select **Kernel → Change Kernel → T2D GPU**.

For an existing environment, install the additional package if necessary:

```bash
conda run -n t2d-gpu python -m pip install xgboost==2.1.4
```

Complete Step 4 before executing this notebook.

## 6. Outputs

Outputs are saved to `Outputs/Step5_XGBoost/`, including the model, run configuration, training history, participant-level predictions, evaluation metrics, and confusion matrices.


In [1]:
# ============================================================
# 0. XGBoost — Configuration and Data Preparation
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import torch
import xgboost as xgb

from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
)

# ------------------------------------------------------------
# 0.1 Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

INPUT_FILE = (
    PROJECT_ROOT
    / "Outputs/Step4_Feature_Assembly/"
      "Multimodal_100_samples_PRSCSx_EqualWeight.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "Outputs/Step5_XGBoost"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = OUTPUT_DIR / "XGBoost_best_model.json"
HISTORY_FILE = OUTPUT_DIR / "XGBoost_TrainingHistory.csv"
CONFIG_FILE = OUTPUT_DIR / "XGBoost_RunConfig.json"
PREDICTION_FILE = OUTPUT_DIR / "XGBoost_Predictions.csv"


# ------------------------------------------------------------
# 0.2 User-defined hyperparameters
# ------------------------------------------------------------

GPU_INDEX = 0
USE_GPU = True
ALLOW_CPU_FALLBACK = True

RANDOM_SEED = 1

N_ESTIMATORS = 100
MAX_DEPTH = 6
LEARNING_RATE = 0.3

REG_ALPHA = 0.0
REG_LAMBDA = 1.0

OBJECTIVE = "binary:logistic"
EVAL_METRIC = "auc"

EARLY_STOPPING_ROUNDS = 30

N_JOBS = 4

# Validation Youden threshold search
THRESHOLD_GRID = np.arange(0.0, 1.0, 0.001)


# ------------------------------------------------------------
# 0.3 Select computational device
# ------------------------------------------------------------

if USE_GPU and torch.cuda.is_available():

    assert GPU_INDEX < torch.cuda.device_count()

    XGB_DEVICE = f"cuda:{GPU_INDEX}"

elif ALLOW_CPU_FALLBACK:

    XGB_DEVICE = "cpu"

else:

    raise RuntimeError("Requested GPU is unavailable.")


# ------------------------------------------------------------
# 0.4 Load assembled participant-level data
# ------------------------------------------------------------

assert INPUT_FILE.is_file(), INPUT_FILE

df = pd.read_csv(
    INPUT_FILE,
    dtype={"Demo_ID": str}
)

META_COLUMNS = ["Demo_ID", "Dataset", "T2D"]

FEATURE_COLUMNS = [
    c for c in df.columns
    if c not in META_COLUMNS
]

assert df.shape == (100, 4103)
assert len(FEATURE_COLUMNS) == 4100
assert df["Demo_ID"].is_unique

df["Dataset"] = df["Dataset"].astype(str).str.lower()

assert df["Dataset"].value_counts().to_dict() == {
    "training1": 32,
    "training2": 32,
    "validation": 16,
    "testing": 20,
}

df["T2D"] = pd.to_numeric(
    df["T2D"],
    errors="raise"
).astype(int)

assert set(df["T2D"].unique()) == {0, 1}

df[FEATURE_COLUMNS] = (
    df[FEATURE_COLUMNS]
    .apply(pd.to_numeric, errors="raise")
    .astype(np.float32)
)

assert np.isfinite(
    df[FEATURE_COLUMNS].to_numpy()
).all()


# ------------------------------------------------------------
# 0.5 Prepare model datasets
# ------------------------------------------------------------

def get_split(split_name):

    subset = df.loc[
        df["Dataset"] == split_name
    ].reset_index(drop=True)

    X = subset[FEATURE_COLUMNS]
    y = subset["T2D"].to_numpy(dtype=int)

    return subset, X, y


df_train, X_train, y_train = get_split("training2")
df_valid, X_valid, y_valid = get_split("validation")
df_test, X_test, y_test = get_split("testing")

assert len(X_train) == 32
assert len(X_valid) == 16
assert len(X_test) == 20

for name, y in [
    ("Training2", y_train),
    ("Validation", y_valid),
    ("Testing", y_test),
]:
    assert len(np.unique(y)) == 2, (
        f"{name} must contain both T2D classes."
    )

print("=" * 65)
print("XGBOOST — INPUT VERIFICATION")
print("=" * 65)

print("XGBoost version:", xgb.__version__)
print("Device:", XGB_DEVICE)
print("Features:", len(FEATURE_COLUMNS))
print("Training2:", X_train.shape)
print("Validation:", X_valid.shape)
print("Testing:", X_test.shape)

print("\nClass distribution:")
display(
    pd.crosstab(
        df["Dataset"],
        df["T2D"]
    ).reindex(
        ["training1", "training2", "validation", "testing"]
    )
)

print("\nXGBoost data preparation completed.")

XGBOOST — INPUT VERIFICATION
XGBoost version: 2.1.4
Device: cuda:0
Features: 4100
Training2: (32, 4100)
Validation: (16, 4100)
Testing: (20, 4100)

Class distribution:


T2D,0,1
Dataset,,
training1,16,16
training2,16,16
validation,8,8
testing,10,10



XGBoost data preparation completed.


In [2]:
# ============================================================
# 1. Train XGBoost
# ============================================================

start_time = time.perf_counter()

model = xgb.XGBClassifier(
    objective=OBJECTIVE,
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    reg_alpha=REG_ALPHA,
    reg_lambda=REG_LAMBDA,
    tree_method="hist",
    device=XGB_DEVICE,
    eval_metric=EVAL_METRIC,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    random_state=RANDOM_SEED,
    n_jobs=N_JOBS,
)

# ------------------------------------------------------------
# 1.1 Training2: fit
# Validation: early stopping
# ------------------------------------------------------------

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False,
)

training_seconds = time.perf_counter() - start_time


# ------------------------------------------------------------
# 1.2 Best iteration selected using Validation AUC
# ------------------------------------------------------------

BEST_ITERATION = int(model.best_iteration)
BEST_N_TREES = BEST_ITERATION + 1

BEST_VALIDATION_AUC = float(model.best_score)


# ------------------------------------------------------------
# 1.3 Save model and training history
# ------------------------------------------------------------

model.save_model(str(MODEL_FILE))

eval_results = model.evals_result()

auc_history = eval_results["validation_0"]["auc"]

history_df = pd.DataFrame({
    "Boosting_round": np.arange(1, len(auc_history) + 1),
    "Validation_AUC": auc_history,
})

history_df.to_csv(HISTORY_FILE, index=False)


# ------------------------------------------------------------
# 1.4 Save configuration
# ------------------------------------------------------------

run_config = {
    "Model": "XGBoost",
    "XGBoost_version": xgb.__version__,
    "Training_split": "training2",
    "Selection_split": "validation",
    "Final_evaluation_split": "testing",
    "Predictors": len(FEATURE_COLUMNS),
    "Objective": OBJECTIVE,
    "Max_depth": MAX_DEPTH,
    "Learning_rate": LEARNING_RATE,
    "Maximum_estimators": N_ESTIMATORS,
    "Reg_alpha": REG_ALPHA,
    "Reg_lambda": REG_LAMBDA,
    "Early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "Evaluation_metric": EVAL_METRIC,
    "GPU_device": XGB_DEVICE,
    "Random_seed": RANDOM_SEED,
    "Best_iteration_zero_based": BEST_ITERATION,
    "Best_n_trees": BEST_N_TREES,
    "Best_validation_AUC": BEST_VALIDATION_AUC,
    "Training_seconds": training_seconds,
}

CONFIG_FILE.write_text(
    json.dumps(run_config, indent=2),
    encoding="utf-8"
)


print("=" * 65)
print("XGBOOST TRAINING COMPLETED")
print("=" * 65)

print("Best boosting round:", BEST_N_TREES)
print("Best validation AUC:", round(BEST_VALIDATION_AUC, 4))
print("Training seconds:", round(training_seconds, 2))
print("Saved model:", MODEL_FILE.relative_to(PROJECT_ROOT))

display(history_df.tail(10))

XGBOOST TRAINING COMPLETED
Best boosting round: 5
Best validation AUC: 0.7969
Training seconds: 4.14
Saved model: Outputs/Step5_XGBoost/XGBoost_best_model.json


,Boosting_round,Validation_AUC
25,26,0.734375
26,27,0.734375
27,28,0.734375
28,29,0.734375
29,30,0.734375
30,31,0.734375
31,32,0.734375
32,33,0.734375
33,34,0.734375
34,35,0.734375


In [3]:
# ============================================================
# 2. Validation Threshold Selection and Predictions
# ============================================================

# ------------------------------------------------------------
# 2.1 Predict using the best boosting iteration
# ------------------------------------------------------------

valid_prob = model.predict_proba(
    X_valid,
    iteration_range=(0, BEST_N_TREES)
)[:, 1]

test_prob = model.predict_proba(
    X_test,
    iteration_range=(0, BEST_N_TREES)
)[:, 1]


# ------------------------------------------------------------
# 2.2 Validation Youden threshold
# ------------------------------------------------------------

threshold_records = []

for threshold in THRESHOLD_GRID:

    valid_pred = (
        valid_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_valid,
        valid_pred,
        labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    threshold_records.append({
        "Threshold": float(threshold),
        "Sensitivity": float(sensitivity),
        "Specificity": float(specificity),
        "Youden_Index": float(
            sensitivity + specificity - 1
        )
    })

threshold_df = pd.DataFrame(threshold_records)

best_threshold_idx = (
    threshold_df["Youden_Index"].to_numpy().argmax()
)

BEST_THRESHOLD = float(
    threshold_df.iloc[best_threshold_idx]["Threshold"]
)

threshold_df.to_csv(
    OUTPUT_DIR / "XGBoost_ValidationThresholdGrid.csv",
    index=False
)


# ------------------------------------------------------------
# 2.3 Build prediction table
# ------------------------------------------------------------

prediction_frames = []

for subset, labels, probabilities, split_name in [
    (df_valid, y_valid, valid_prob, "validation"),
    (df_test, y_test, test_prob, "testing"),
]:

    result = pd.DataFrame({
        "Demo_ID": subset["Demo_ID"].to_numpy(),
        "Dataset": split_name,
        "T2D": labels,
        "Predicted_Probability": probabilities,
        "Predicted_Label": (
            probabilities >= BEST_THRESHOLD
        ).astype(int),
        "Threshold": BEST_THRESHOLD,
    })

    prediction_frames.append(result)

prediction_df = pd.concat(
    prediction_frames,
    ignore_index=True
)

assert len(prediction_df) == 36
assert prediction_df["Demo_ID"].is_unique

prediction_df.to_csv(
    PREDICTION_FILE,
    index=False
)


# ------------------------------------------------------------
# 2.4 Display
# ------------------------------------------------------------

run_config["Validation_Youden_Threshold"] = BEST_THRESHOLD

CONFIG_FILE.write_text(
    json.dumps(run_config, indent=2),
    encoding="utf-8"
)

print("=" * 65)
print("XGBOOST PREDICTIONS COMPLETED")
print("=" * 65)

print("Validation-selected threshold:", BEST_THRESHOLD)
print("Validation N:", len(valid_prob))
print("Testing N:", len(test_prob))

print("\nPredictions:")
display(prediction_df.head(10))


# Variables for the shared evaluation cell
EVAL_MODEL_NAME = "XGBoost"
EVAL_OUTPUT_DIR = OUTPUT_DIR
EVAL_PREDICTIONS = prediction_df
EVAL_THRESHOLD = BEST_THRESHOLD

[LOCAL_PATH] UserWarning: [10:26:29] WARNING: [LOCAL_PATH] Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


XGBOOST PREDICTIONS COMPLETED
Validation-selected threshold: 0.5720000000000001
Validation N: 16
Testing N: 20

Predictions:


,Demo_ID,Dataset,T2D,Predicted_Probability,Predicted_Label,Threshold
0,D019,validation,1,0.774219,1,0.572
1,D028,validation,1,0.820478,1,0.572
2,D035,validation,0,0.461756,0,0.572
3,D038,validation,0,0.692116,1,0.572
4,D042,validation,0,0.571926,0,0.572
5,D043,validation,1,0.619298,1,0.572
6,D044,validation,1,0.694316,1,0.572
7,D057,validation,1,0.495412,0,0.572
8,D058,validation,0,0.313286,0,0.572
9,D065,validation,1,0.344554,0,0.572
